In [27]:
# ============================================
# Product Demand Forecasting
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib

print("Libraries imported successfully.")

Libraries imported successfully.


In [28]:
# Load Dataset

df = pd.read_csv("feature_engineered_data.csv")

print(df.shape)

df.head()

(779425, 19)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue,Year,Month,MonthName,Day,DayOfWeek,Hour,Quarter,IsWeekend,BasketSize,OrderValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,12,December,1,Tuesday,7,4,0,166,505.3
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,Tuesday,7,4,0,166,505.3
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,12,December,1,Tuesday,7,4,0,166,505.3
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,12,December,1,Tuesday,7,4,0,166,505.3
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,12,December,1,Tuesday,7,4,0,166,505.3


In [29]:
df = pd.read_csv("feature_engineered_data.csv")

print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Revenue', 'Year', 'Month', 'MonthName', 'Day', 'DayOfWeek', 'Hour', 'Quarter', 'IsWeekend', 'BasketSize', 'OrderValue']


In [30]:
df = df[
[
    "StockCode",
    "InvoiceDate",
    "Quantity",
    "Revenue",
    "OrderValue",
    "BasketSize",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "IsWeekend"
]
]

print(df.head())

  StockCode          InvoiceDate  Quantity  Revenue  OrderValue  BasketSize  \
0     85048  2009-12-01 07:45:00        12     83.4       505.3         166   
1    79323P  2009-12-01 07:45:00        12     81.0       505.3         166   
2    79323W  2009-12-01 07:45:00        12     81.0       505.3         166   
3     22041  2009-12-01 07:45:00        48    100.8       505.3         166   
4     21232  2009-12-01 07:45:00        24     30.0       505.3         166   

   Year  Month  Day DayOfWeek  IsWeekend  
0  2009     12    1   Tuesday          0  
1  2009     12    1   Tuesday          0  
2  2009     12    1   Tuesday          0  
3  2009     12    1   Tuesday          0  
4  2009     12    1   Tuesday          0  


In [31]:
product_df = (
    df.groupby(["StockCode","InvoiceDate"], as_index=False)
      .agg({
          "Quantity":"sum",
          "Revenue":"sum",
          "OrderValue":"sum",
          "BasketSize":"mean",
          "Year":"first",
          "Month":"first",
          "Day":"first",
          "DayOfWeek":"first",
          "IsWeekend":"first"
      })
)

product_df = product_df.sort_values(["StockCode","InvoiceDate"])

print(product_df.head())

  StockCode          InvoiceDate  Quantity  Revenue  OrderValue  BasketSize  \
0     10002  2009-12-01 09:08:00        12    10.20      310.75       145.0   
1     10002  2009-12-03 13:49:00         1     0.85      204.30       113.0   
2     10002  2009-12-03 19:13:00         1     0.85      159.95        69.0   
3     10002  2009-12-03 20:03:00         4     3.40      407.22       227.0   
4     10002  2009-12-04 08:46:00        12    10.20      337.10       152.0   

   Year  Month  Day DayOfWeek  IsWeekend  
0  2009     12    1   Tuesday          0  
1  2009     12    3  Thursday          0  
2  2009     12    3  Thursday          0  
3  2009     12    3  Thursday          0  
4  2009     12    4    Friday          0  


In [32]:
# Create Previous Demand Features

product_df["Lag1"] = product_df.groupby("StockCode")["Quantity"].shift(1)
product_df["Lag2"] = product_df.groupby("StockCode")["Quantity"].shift(2)
product_df["Lag3"] = product_df.groupby("StockCode")["Quantity"].shift(3)

product_df = product_df.dropna()

print(product_df.head())
print(product_df.shape)

  StockCode          InvoiceDate  Quantity  Revenue  OrderValue  BasketSize  \
3     10002  2009-12-03 20:03:00         4     3.40      407.22       227.0   
4     10002  2009-12-04 08:46:00        12    10.20      337.10       152.0   
5     10002  2009-12-04 12:20:00        12    10.20     1185.16       713.0   
6     10002  2009-12-04 13:47:00        48    40.80      435.88       504.0   
7     10002  2009-12-04 17:31:00         1     0.85      205.35       199.0   

   Year  Month  Day DayOfWeek  IsWeekend  Lag1  Lag2  Lag3  
3  2009     12    3  Thursday          0   1.0   1.0  12.0  
4  2009     12    4    Friday          0   4.0   1.0   1.0  
5  2009     12    4    Friday          0  12.0   4.0   1.0  
6  2009     12    4    Friday          0  12.0  12.0   4.0  
7  2009     12    4    Friday          0  48.0  12.0  12.0  
(753884, 14)


In [33]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

product_df["StockCodeEncoded"] = encoder.fit_transform(product_df["StockCode"])

print(product_df.head())

  StockCode          InvoiceDate  Quantity  Revenue  OrderValue  BasketSize  \
3     10002  2009-12-03 20:03:00         4     3.40      407.22       227.0   
4     10002  2009-12-04 08:46:00        12    10.20      337.10       152.0   
5     10002  2009-12-04 12:20:00        12    10.20     1185.16       713.0   
6     10002  2009-12-04 13:47:00        48    40.80      435.88       504.0   
7     10002  2009-12-04 17:31:00         1     0.85      205.35       199.0   

   Year  Month  Day DayOfWeek  IsWeekend  Lag1  Lag2  Lag3  StockCodeEncoded  
3  2009     12    3  Thursday          0   1.0   1.0  12.0                 0  
4  2009     12    4    Friday          0   4.0   1.0   1.0                 0  
5  2009     12    4    Friday          0  12.0   4.0   1.0                 0  
6  2009     12    4    Friday          0  12.0  12.0   4.0                 0  
7  2009     12    4    Friday          0  48.0  12.0  12.0                 0  


In [37]:
from sklearn.preprocessing import LabelEncoder

day_encoder = LabelEncoder()

product_df["DayOfWeek"] = day_encoder.fit_transform(product_df["DayOfWeek"])

print(product_df[["DayOfWeek"]].head())

   DayOfWeek
3          4
4          0
5          0
6          0
7          0


In [38]:
features = [
    "StockCodeEncoded",
    "Revenue",
    "OrderValue",
    "BasketSize",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "IsWeekend",
    "Lag1",
    "Lag2",
    "Lag3"
]

X = product_df[features]
y = product_df["Quantity"]

print(X.shape)
print(y.shape)

(753884, 12)
(753884,)


In [39]:
split = int(len(product_df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(X_train.shape)
print(X_test.shape)

(603107, 12)
(150777, 12)


In [40]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=30,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [42]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE :", rmse)
print("R2 Score :", r2)

MAE : 7.9370745137503755
RMSE : 64.78666823014808
R2 Score : 0.5253577552154035


In [43]:
import joblib

joblib.dump(model, "product_demand_model.pkl")
joblib.dump(encoder, "stockcode_encoder.pkl")

print("Model Saved Successfully")

Model Saved Successfully


In [44]:
prediction_df = X_test.copy()

prediction_df["Actual_Demand"] = y_test.values
prediction_df["Predicted_Demand"] = y_pred

prediction_df.to_csv("product_demand_predictions.csv", index=False)

print("Prediction CSV Saved Successfully")

Prediction CSV Saved Successfully


In [45]:
import os

size = os.path.getsize("product_demand_model.pkl") / (1024 * 1024)

print(f"Model Size : {size:.2f} MB")

Model Size : 1.43 MB


In [46]:
joblib.dump(day_encoder, "dayofweek_encoder.pkl")

print("DayOfWeek Encoder Saved Successfully")

DayOfWeek Encoder Saved Successfully
